# AF-CALVADOS


# Import and functions

In [ ]:
# Code from AF-CALVADOS notebook
!pip install openmm[cuda12]
!pip install git+https://github.com/KULL-Centre/CALVADOS.git@colab_compatible
!pip install py3Dmol

import os
import calvados as cal
from calvados.cfg import Config, Job, Components, load_ebi
from calvados import sim
import subprocess
import numpy as np
import numba as nb
import pandas as pd
import math
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import MDAnalysis as mda
from MDAnalysis.coordinates.PDB import PDBWriter
from MDAnalysis.analysis import align

from io import StringIO
import json
import requests
from pathlib import Path

import py3Dmol

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams['font.size'] = 6
plt.rcParams['ytick.major.size'] = 2
plt.rcParams['ytick.minor.size'] = 1
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['xtick.minor.size'] = 1
plt.rcParams['xtick.labelsize'] = 6
plt.rcParams['ytick.labelsize'] = 6
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['axes.labelpad'] = 2
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['figure.dpi'] = 300
plt.rcParams['lines.markersize'] = 3
plt.rcParams['lines.markeredgewidth'] = 0.5
plt.rcParams['lines.linewidth'] = 1.
plt.rcParams['figure.max_open_warning'] = 30
col1x = 3.425 # column
col2x = 7. # 2x column
github_folder = 'https://raw.githubusercontent.com/KULL-Centre/CALVADOS/colab_compatible'

cwd = os.getcwd()

def load_pae(name, pdb_folder):
    input_pae = f'{pdb_folder}/{name}.json'
    pae = cal.build.load_pae(input_pae,symmetrize=True,colabfold=0) / 10. # nm
    return pae

def calc_rgs(name, path, start=10,stop=None,step=1):
    u = mda.Universe(f'{path}/top.pdb',f'{path}/{name}.dcd')
    ag = u.atoms
    rg = cal.analysis.calc_rg(u, ag, start=start, stop=stop, step=step)
    return rg

def calc_ees(name, path, start=10,stop=None,step=1):
    u = mda.Universe(f'{path}/top.pdb',f'{path}/{name}.dcd')
    ag = u.atoms
    ees, _, _ = cal.analysis.calc_ete(u,ag,start=start,stop=stop,step=step)
    return ees

def calc_timesteps(name, path, start=10, stop=None, step=1):
    u = mda.Universe(f'{path}/top.pdb',f'{path}/{name}.dcd')
    dt = u.trajectory[0].dt * step
    xs = np.arange(len(u.trajectory[start:stop:step])) * dt / 1000.
    return xs

def calc_dmap_std_manual(name, path, start=10,stop=None,step=1):
    u = mda.Universe(f'{path}/top.pdb',f'{path}/{name}.dcd')
    ag = u.atoms

    box = u.dimensions[:3] / 10.
    coordinates = u.trajectory.timeseries(ag) / 10.

    nres = len(ag)

    dmap_std_manual = calc_dmap_std_numba(coordinates,box,nres,start=start,stop=stop,step=step)
    return dmap_std_manual

@staticmethod
@nb.jit(nopython=True)
def calc_dmap_std_numba(coordinates,box,nres,start=10,stop=None,step=1):
    dmaps_std_manual = np.zeros((nres,nres))

    for idx in range(nres):
        for jdx in range(idx,nres):
            x0 = coordinates[idx,start:stop:step]
            x1 = coordinates[jdx,start:stop:step]
            dvec = x1-x0
            dvec_pbc = np.where(dvec<=box/2, dvec, box-dvec)
            d = np.zeros((len(x0)))
            for kdx, dv_pbc in enumerate(dvec_pbc):
                d[kdx] = math.sqrt(dv_pbc[0]**2 + dv_pbc[1]**2 + dv_pbc[2]**2)
            d_std = np.std(d)

            dmaps_std_manual[idx,jdx] = d_std
            dmaps_std_manual[jdx,idx] = d_std
    return dmaps_std_manual

def load_plddt(name, pdb_folder):
    # pLDDT
    pdb = f'{pdb_folder}/{name}.pdb'
    plddt = cal.build.bfac_from_pdb(pdb)
    # bfac_matrix = np.minimum.outer(bfac,bfac)
    return plddt

################

def plot_rg(name, path, start=10, stop=None, step=1):
    rgs = calc_rgs(name, path, start=start,stop=stop,step=step)
    fig, ax = plt.subplots(1,2,figsize=(5,2))

    xs = calc_timesteps(name, path, start=start)
    ax[0].plot(xs, rgs)
    ax[0].set(xlim=(xs[0],xs[-1]))
    ax[0].set(xlabel='Time [ns]', ylabel=r'$R_\mathrm{g}$ [nm]')

    ax[1].hist(rgs,bins=50,histtype='step',color='C0',density=True)
    ax[1].hist(rgs,bins=50,alpha=0.3,color='C0',density=True)
    ax[1].set(xlabel=r'$R_\mathrm{g}$ [nm]', ylabel='pdf')

    fig.tight_layout()
    os.makedirs(f'{name}/analysis',exist_ok=True)
    np.savetxt(f'{name}/analysis/{name}_rg.txt', rgs.T)
    fig.savefig(f'{name}/analysis/{name}_rg.pdf')

def plot_ee(name, path, start=10, stop=None, step=1):
    # rgs = calc_rgs(name, path, start=10,stop=None,step=1)
    ees = calc_ees(name, path, start=start,stop=stop,step=step)
    fig, ax = plt.subplots(1,2,figsize=(5,2))

    xs = calc_timesteps(name, path, start=start)
    ax[0].plot(xs, ees)
    ax[0].set(xlim=(xs[0],xs[-1]))
    ax[0].set(xlabel='Time [ns]', ylabel=r'$R_\mathrm{ee}$ [nm]')

    ax[1].hist(ees,bins=50,histtype='step',color='C0',density=True)
    ax[1].hist(ees,bins=50,alpha=0.3,color='C0',density=True)
    ax[1].set(xlabel=r'$R_\mathrm{ee}$ [nm]', ylabel='pdf')

    fig.tight_layout()
    os.makedirs(f'{name}/analysis',exist_ok=True)
    np.savetxt(f'{name}/analysis/{name}_ree.txt', ees.T)
    fig.savefig(f'{name}/analysis/{name}_ree.pdf')

def plot_plddt(name, pdb_folder):
    seq = cal.sequence.seq_from_pdb(f'{pdb_folder}/{name}.pdb')[0]
    print(seq)
    nres = len(seq)
    xs = np.arange(1,nres+1)

    plddt = load_plddt(name, pdb_folder)
    fig, ax = plt.subplots(figsize=(3,1.5))
    ax.plot(xs, plddt*100)
    ax.set(xlim=(xs[0],xs[-1]))
    ax.set(xlabel='Residue', ylabel='pLDDT')
    fig.tight_layout()
    os.makedirs(f'{name}/analysis',exist_ok=True)
    with open(f'{name}/analysis/{name}_pLDDT.txt','w') as f:
        for x, pl in zip(xs, plddt):
            f.write(f'{x} {pl}\n')
    fig.savefig(f'{name}/analysis/{name}_pLDDT.pdf')

def plot_pae_sigma(name, path, pdb_folder, start=10, stop=None, step=1):
    pae = load_pae(name, pdb_folder)
    nres = len(pae)
    fig, ax = plt.subplots(1,2,figsize=(4,2.2))
    axij_pae = ax[0]
    _ = axij_pae.imshow(pae,cmap=plt.cm.Oranges_r,vmin=0,vmax=3.)

    divider = make_axes_locatable(axij_pae)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    # plt.colorbar(im, cax=cax)
    plt.colorbar(_,cax=cax,label='nm')

    axij_pae.set_title(f'{name}\nPAE', fontsize=6, linespacing=1.5)

    xs = np.arange(0,nres,100)

    cmap = plt.cm.Blues_r

    dmaps_std = calc_dmap_std_manual(name, path, start=start, stop=stop, step=step)

    axij_dmaps = ax[1]

    _ = axij_dmaps.imshow(dmaps_std,cmap=cmap,vmin=0,vmax=2)
    axij_dmaps.set_title(f'{name}\n'+r'$\sigma(r)$ Sim.', fontsize=6, linespacing=1.5)
    divider = make_axes_locatable(axij_dmaps)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    # plt.colorbar(im, cax=cax)
    plt.colorbar(_,cax=cax,label='nm')

    for a in [axij_pae, axij_dmaps]:
        a.grid(False)
        a.set(xlabel='Residue',ylabel='Residue')
    fig.tight_layout()
    os.makedirs(f'{name}/analysis',exist_ok=True)
    np.save(f'{name}/analysis/{name}_PAE.npy', pae)
    np.save(f'{name}/analysis/{name}_sigma.npy', dmaps_std)
    fig.savefig(f'{name}/analysis/{name}_PAE_vs_sigma.pdf')

##############

def set_up_system(name, path, temp, ionic, pH, L, N_save, N_frames, pdb_folder, fresidues):

    config = Config(
      # GENERAL
      sysname = name, # name of simulation system
      box = [L, L, L], # nm
      temp = temp, # 20 degrees Celsius
      ionic = ionic, # molar
      pH = pH, # 7.5
      topol = 'center',

      # RUNTIME SETTINGS
      wfreq = N_save, # dcd writing interval, 1 = 10 fs
      steps = N_frames*N_save, # number of simulation steps
      runtime = 0, # overwrites 'steps' keyword if > 0
      platform = 'CUDA', # or CUDA
      restart = 'checkpoint',
      frestart = 'restart.chk',
      verbose = True,
    )

    # PATH
    subprocess.run(f'mkdir -p {path}',shell=True)

    config.write(path,name='config.yaml')

    components = Components(
      # Defaults
      molecule_type = 'protein',
      nmol = 1, # number of molecules
      restraint = True, # apply restraints
      charge_termini = 'both', # charge N or C or both

      # INPUT
      fresidues = fresidues, # residue definitions
      pdb_folder = pdb_folder, # directory for pdb and PAE files

      # RESTRAINTS
      restraint_type = 'go', # harmonic or go
      use_com = True, # apply on centers of mass instead of CA
      colabfold = 0, # PAE format (EBI AF=0, Colabfold=1&2)
      k_go = 15., # Restraint force constant
    )
    components.add(name=name)
    components.write(path,name='components.yaml')

####################

ALPHAFOLD_API = "https://alphafold.ebi.ac.uk/api/prediction"

def write_entry(uniprot, entry, pdb_folder):
    out = Path(pdb_folder) / f"{uniprot}_info.json"
    with open(out, 'w') as f:
        json.dump(entry, f, indent=2)

def download_file(url, dest):
    r = requests.get(url, allow_redirects=True)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        f.write(r.content)

def load_ebi(uniprot, pdb_folder):
    folder = Path(pdb_folder)
    folder.mkdir(parents=True, exist_ok=True)

    # Fetch entry metadata
    r = requests.get(f"{ALPHAFOLD_API}/{uniprot}")
    r.raise_for_status()
    results = r.json()

    if not results:
        raise ValueError(f"No AlphaFold entry found for UniProt ID: {uniprot}")

    entry = results[0]

    # Download PDB structure
    pdb_url = entry.get("pdbUrl")
    if not pdb_url:
        raise KeyError(f"'pdbUrl' not found in API response for {uniprot}")
    download_file(pdb_url, folder / f"{uniprot}.pdb")

    # Download PAE JSON — prefer new field, fall back to deprecated one
    pae_url = entry.get("paeDocUrl")
    if not pae_url:
        raise KeyError(f"No PAE doc URL found in API response for {uniprot}")
    download_file(pae_url, folder / f"{uniprot}.json")

    # Save full entry metadata
    write_entry(uniprot, entry, pdb_folder)
    print(f"Done: {uniprot}")

##############


def align_traj(pdb, traj, start=None, stop=None, step=1, reference_frame=0):
    """RMSD-align trajectory to a reference frame and write a new DCD"""

    u = mda.Universe(pdb, traj)

    # Reference structure (usually first frame)
    ref = mda.Universe(pdb, traj)
    ref.trajectory[reference_frame]

    with mda.Writer(f"{traj[:-4]}_aligned.dcd", len(u.atoms)) as W:
        for ts in u.trajectory[start:stop:step]:
            # Align current frame to reference
            align.alignto(u, ref, select="all")
            W.write(u.atoms)

#########

def postprocess_traj(name, path, step_view = 10):
    cal.analysis.center_traj(f'{path}/top.pdb', f'{path}/{name}.dcd', start=10)
    align_traj(f'{path}/top.pdb', f'{path}/{name}_c.dcd')

    u = mda.Universe(f'{path}/top.pdb', f'{path}/{name}_c_aligned.dcd')

    nres = len(u.atoms)
    models = []
    pdb_io = StringIO()

    with mda.Writer('tmp.pdb', multiframe=True, format='pdb') as W:
        for ts in u.trajectory[::step_view]:
            W.write(u.atoms)
    return nres

def visualize(nres):
    cmap = plt.cm.viridis
    view = py3Dmol.view(width=800, height=600)
    with open('tmp.pdb', 'r') as f:
        pdb_str = f.read()
        view.addModelsAsFrames(pdb_str)

    view.setStyle({"sphere": {}})
    for idx in range(nres):
        color = np.array(cmap(idx/nres)[:3])*255
        colorstr = f'rgb({color[0]}, {color[1]}, {color[2]})'
        view.setStyle({"serial": [idx+1]}, {"sphere": {"color": colorstr}})

    view.zoomTo()
    view.animate({"loop": "forward"})
    view.show()

In [ ]:
# Adapted functions for plotting
def plot_plddt_with_motifs(name, pdb_folder, idr_regions=None, motifs=None):
    """Adapted plddt visualization with added highlighting of SLiMs/conserved IDRs"""
    seq = cal.sequence.seq_from_pdb(f'{pdb_folder}/{name}.pdb')[0]
    nres = len(seq)
    xs = np.arange(1, nres+1)
    plddt = load_plddt(name, pdb_folder)
    fig, ax = plt.subplots(figsize=(4, 2))
    ax.plot(xs, plddt*100, color='black', linewidth=1)
    # Highlight IDRs
    if idr_regions:
        for i, (idr_start, idr_end) in enumerate(idr_regions):
            ax.axvspan(idr_start, idr_end,
                      color='lightblue', alpha=0.4,
                      label='IDR' if i == 0 else "")
    # Highlight motifs
    if motifs:
        for i, (m_start, m_end) in enumerate(motifs):
            ax.axvspan(m_start, m_end,
                      color='orange', alpha=0.4,
                      label='Motif' if i == 0 else "") 
    ax.legend(fontsize=5, loc='upper right')        
    ax.set(xlim=(xs[0], xs[-1]), xlabel='Residue', ylabel='pLDDT', title=f'{name} confidence')
    fig.tight_layout()
    fig.savefig(f'{name}/analysis/{name}_pLDDT_motifs.pdf')

def plot_pae_sigma_with_motifs(name, path, pdb_folder, idr_regions=None,
                               motifs=None, start=10, stop=None, step=1):
    """Adapted pae_sigma visualization with added highlighting of SLiMs/conserved IDRs"""
    pae = load_pae(name, pdb_folder)
    nres = len(pae)
    fig, ax = plt.subplots(1,2,figsize=(4,2.2))
    # PAE Plot
    axij_pae = ax[0]
    im0 = axij_pae.imshow(pae, cmap=plt.cm.Oranges_r, vmin=0, vmax=3.)
    divider0 = make_axes_locatable(axij_pae)
    cax0 = divider0.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im0, cax=cax0, label='nm')
    axij_pae.set_title(f'{name}\nPAE', fontsize=6, linespacing=1.5)
    # Simulation Variance Plot (sigma) ---
    dmaps_std = calc_dmap_std_manual(name, path, start=start, stop=stop, step=step)
    axij_dmaps = ax[1]
    im1 = axij_dmaps.imshow(dmaps_std, cmap=plt.cm.Blues_r, vmin=0, vmax=2)
    axij_dmaps.set_title(f'{name}\n'+r'$\sigma(r)$ Sim.', fontsize=6, linespacing=1.5)
    divider1 = make_axes_locatable(axij_dmaps)
    cax1 = divider1.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(im1, cax=cax1, label='nm')
    # IDR highlighting
    if idr_regions:
        for (idr_start, idr_end) in idr_regions:
            for axis in [axij_pae, axij_dmaps]:
                axis.axvspan(idr_start, idr_end,
                            color='grey', alpha=0.25, lw=0)
                axis.axhspan(idr_start, idr_end,
                            color='grey', alpha=0.25, lw=0)
    # Motif highlighting
    if motifs:
        for (m_start, m_end) in motifs:
            for axis in [axij_pae, axij_dmaps]:
                axis.axvspan(m_start, m_end, color='black', alpha=0.3, lw=0)
                axis.axhspan(m_start, m_end, color='black', alpha=0.3, lw=0)
    for a in [axij_pae, axij_dmaps]:
        a.grid(False)
        a.set(xlabel='Residue', ylabel='Residue')
        
    fig.tight_layout()
    os.makedirs(f'{name}/analysis', exist_ok=True)
    np.save(f'{name}/analysis/{name}_PAE.npy', pae)
    np.save(f'{name}/analysis/{name}_sigma.npy', dmaps_std)
    fig.savefig(f'{name}/analysis/{name}_PAE_vs_sigma.pdf')
    plt.close()
    return dmaps_std 


def visualize_with_motifs(nres, motifs=None):
    """3D visualization with added highlighting of SLiMs"""
    cmap = plt.cm.viridis
    view = py3Dmol.view(width=800, height=600)
    with open('tmp.pdb', 'r') as f:
        pdb_str = f.read()
        view.addModelsAsFrames(pdb_str)
    for idx in range(nres):
        res_num = idx + 1
        # Check if this residue is part of any motif
        is_motif = any(start <= res_num <= end for start, end in motifs) if motifs else False
        if is_motif:
            colorstr = 'rgb(255, 0, 0)' # Bright Red for Motifs
        else:
            color = np.array(cmap(idx/nres)[:3])*255
            colorstr = f'rgb({color[0]}, {color[1]}, {color[2]})'
        view.setStyle({"serial": [res_num]}, {"sphere": {"color": colorstr, "radius": 1.5}})

    view.zoomTo()
    view.show()
    # didn't run but to save the file to html
    view_html = view._make_html()
    with open(f"{name}_3d_view.html", "w") as f:
        f.write(view_html)


def export_disordered_lowpae_residues(name, pdb_folder, path,
                                       idr_regions=None,
                                       motifs=None,
                                       pae_threshold=1.0,
                                       start=10, stop=None, step=1):
    """
    For residues belonging to defined IDR regions, extracts:
      - pLDDT score
      - Mean and min PAE vs all other IDR residues
      - Mean and min sigma(r) vs all other IDR residues
      - Whether PAE < pae_threshold
      - Whether the residue overlaps with a known motif
    """
    plddt = load_plddt(name, pdb_folder)
    pae = load_pae(name, pdb_folder)
    seq = cal.sequence.seq_from_pdb(f'{pdb_folder}/{name}.pdb')[0]
    nres = len(seq)
    residue_numbers = np.arange(1, nres + 1)
    # Load sigma(r) matrix
    sigma_path = f'{name}/analysis/{name}_sigma.npy'
    if os.path.exists(sigma_path):
        dmaps_std = np.load(sigma_path)
    else:
        print(f"  Sigma matrix not found at {sigma_path}, computing now...")
        dmaps_std = calc_dmap_std_manual(name, path, start=start,
                                          stop=stop, step=step)
    idr_residue_set = set()
    idr_label_map = {}  # residue_number -> which IDR(s) it belongs to
    if idr_regions:
        for (idr_start, idr_end) in idr_regions:
            for res in range(idr_start, idr_end + 1):
                idr_residue_set.add(res)
                if res not in idr_label_map:
                    idr_label_map[res] = []
                idr_label_map[res].append(f"{idr_start}-{idr_end}")
    idr_indices = np.array([r - 1 for r in idr_residue_set if 0 <= r - 1 < nres], dtype=int)
    idr_indices = np.sort(idr_indices)
    all_indices = np.arange(nres)
    core_indices = np.array([i for i in all_indices if (i + 1) not in idr_residue_set])
    if len(idr_indices) == 0:
        print(f"  Warning: no IDR residues found for {name}")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    # Build motif residue lookup
    motif_residue_set = set()
    motif_label_map = {}
    if motifs:
        for (m_start, m_end) in motifs:
            for res in range(m_start, m_end + 1):
                motif_residue_set.add(res)
                if res not in motif_label_map:
                    motif_label_map[res] = []
                motif_label_map[res].append(f"{m_start}-{m_end}")
    # Compute metrics for each IDR residue
    records = []
    for idx in idr_indices:
        res_num = residue_numbers[idx]
        # --- LOCAL METRICS (vs IDR) ---
        # PAE vs all other IDR residues (exclude self)
        pae_to_idr = pae[idx, idr_indices]
        pae_to_idr = pae_to_idr[idr_indices != idx]
        mean_pae_idr = np.mean(pae_to_idr) if len(pae_to_idr) > 0 else np.nan
        min_pae_idr  = np.min(pae_to_idr)  if len(pae_to_idr) > 0 else np.nan
        # Sigma(r) vs all other IDR residues (exclude self diagonal = 0)
        sigma_to_idr = dmaps_std[idx, idr_indices]
        sigma_to_idr = sigma_to_idr[idr_indices != idx]
        mean_sigma_idr = np.mean(sigma_to_idr) if len(sigma_to_idr) > 0 else np.nan
        min_sigma_idr  = np.min(sigma_to_idr)  if len(sigma_to_idr) > 0 else np.nan
        # --- GLOBAL METRICS (vs Core) ---
        if len(core_indices) > 0:
            pae_to_core = pae[idx, core_indices]
            mean_pae_core = np.mean(pae_to_core)
            min_pae_core  = np.min(pae_to_core) 
            sigma_to_core = dmaps_std[idx, core_indices]
            mean_sigma_core = np.mean(sigma_to_core)
            min_sigma_core  = np.min(sigma_to_core)
        else:
            mean_pae_core = min_pae_core = mean_sigma_core = np.nan
        # IDR membership label
        idr_label = "; ".join(idr_label_map.get(res_num, ["unknown"]))
        # Motif overlap
        in_motif = res_num in motif_residue_set
        motif_coords = "; ".join(motif_label_map[res_num]) if in_motif else "None"
        records.append({
            'protein_name': name,
            'residue_number': res_num,
            'amino_acid': seq[idx],
            'pLDDT': round(plddt[idx] * 100, 2),
            'IDR_coordinates': idr_label,
            # IDR-Specific Metrics
            'mean_PAE_vs_IDR': round(mean_pae_idr, 4),
            'min_PAE_vs_IDR': round(min_pae_idr, 4),
            'mean_sigma_vs_IDR': round(mean_sigma_idr, 4),
            'min_sigma_vs_IDR': round(min_sigma_idr, 4),
            # Core-Specific Metrics
            'mean_PAE_vs_Core': round(mean_pae_core, 4),
            'min_PAE_vs_Core': round(min_pae_core, 4),
            'mean_sigma_vs_Core': round(mean_sigma_core, 4),
            'min_sigma_vs_Core': round(min_sigma_core, 4),
            'flag_fixed_to_IDR': min_pae_idr < pae_threshold,
            'flag_fixed_to_Core': min_pae_core < pae_threshold,
            'known_motif_overlap': in_motif,
            'motif_coordinates': motif_coords
        })
    df_all_idr = pd.DataFrame(records)
    df_low_pae = df_all_idr[df_all_idr['flag_fixed_to_IDR'] | df_all_idr['flag_fixed_to_Core']].copy()
    df_motif_overlap = df_all_idr[
        (df_all_idr['flag_fixed_to_IDR'] | df_all_idr['flag_fixed_to_Core']) & 
        df_all_idr['known_motif_overlap']
    ].copy()
    os.makedirs(f'{name}/analysis', exist_ok=True)
    df_all_idr.to_csv(
        f'{name}/analysis/{name}_all_IDR_residues.csv', index=False)
    df_low_pae.to_csv(
        f'{name}/analysis/{name}_IDR_lowPAE_residues.csv', index=False)
    df_motif_overlap.to_csv(
        f'{name}/analysis/{name}_IDR_lowPAE_motif_overlap.csv', index=False)
    print(f"\n{'='*50}")
    print(f"Protein: {name}")
    print(f"  Total residues: {nres}")
    print(f"  IDR residues (from dictionary): {len(idr_indices)}")
    print(f"  IDR + low PAE (< {pae_threshold} nm): {len(df_low_pae)}")
    print(f"  IDR + low PAE + known motif overlap: {len(df_motif_overlap)}")
    print(f"{'='*50}\n")
    return df_all_idr, df_low_pae, df_motif_overlap

# Input and main analysis

In [ ]:
protein_data = {
    "Lsr2": "P9WIP7",
    "EspD": "P9WJD5",
    "EspA": "P9WJE1",
    "EspM": "P96214",
    "EspE": "P9WJD3",
    "EspH": "O69732",
    "PPE68": "P9WHW9",
    "EspI": "P9WJC5",
    "EspJ": "P9WJC3",
    "EspK": "P9WJC1",
    "EspB": "P9WJD9"
}

idr_data = {
    "Lsr2": [(61, 76)],
    "EspD": [(1, 14), (35, 60), (171, 184)],
    "EspA": [(270, 294), (301, 383)],
    "EspM": [(1, 34), (131, 161), (230, 249)],
    "EspE": [(204, 277), (361, 402)],
    "EspH": [(1, 33)],
    "PPE68": [(173, 202), (236, 368)],
    "EspI": [(1, 246), (257, 311), (325, 373)],
    "EspJ": [(97, 113), (176, 280)],
    "EspK": [(175, 461)],
    "EspB": [(93, 124), (265, 335), (352, 433), (453, 460)]
}

motif_data = {
    'PPE68': [(341, 343), (265, 267)],
    'EspA': [(346, 348), (279, 281)],
    'EspB': [(456, 460), (456, 460), (459, 460), (384, 386)],
    'EspE': [(368, 370)],
    'EspI': [(57, 59), (206, 208), (217, 219), (286, 288), (351, 353), (1, 4)],
    'EspJ': [(196, 198), (257, 259), (102, 104)],
    'EspK': [(252, 254), (329, 331)],
    'EspM': [(146, 148), (246, 248), (1, 4)]
}
fresidues = '/kaggle/input/datasets/ms1089/residues-csv/residues.csv'
name = 'O75764' #@param {type:"string"}
temp = 310 #@param {type:"number"}
ionic = 0.15 #@param {type:"number"}
pH = 7.4 #@param {type:"number"}

step_size_ns = 0.01 #@param {type:"number", label:"Step size [ns]"}
sim_length_ns = 10. #@param {type:"number", label:"Simulation length [ns]"}

# set the side length of the cubic box
L = 80

# set the saving interval (number of integration steps)
N_save = int(step_size_ns * 100 * 1000)
N_frames = int(sim_length_ns / step_size_ns)
print(f'wfreq: {N_save}, frames: {N_frames}')
N_frames += 10
for protein, ID in protein_data.items():
    os.makedirs(f'{cwd}/{protein}/input',exist_ok=True)
    pdb_folder = f'{cwd}/{protein}/input'
    path = f'{cwd}/{protein}/simulation'
    current_motifs = motif_data.get(protein, [])
    current_idrs   = idr_data.get(protein, [])
    load_ebi(ID, pdb_folder)
    set_up_system(ID, path, temp, ionic, pH, L, N_save, N_frames, pdb_folder, fresidues)
    # simulation
    print(f"Starting simulation for {protein} ({ID})...")
    sim.run(path=path,fconfig='config.yaml',fcomponents='components.yaml')
    # visualization
    nres = postprocess_traj(ID, path)
    # visualize(nres)
    plot_rg(ID, path)
    plot_ee(ID, path)
    print(f"Generating motif-highlighted plots for {protein}...")
    plot_plddt_with_motifs(ID, pdb_folder, idr_regions=current_idrs, motifs=current_motifs)
    plot_pae_sigma_with_motifs(ID, path, pdb_folder, idr_regions=current_idrs, motifs=current_motifs)
    visualize_with_motifs(nres, motifs=current_motifs)
    df_all, df_low_pae, df_motif = export_disordered_lowpae_residues(
        name=ID,
        pdb_folder=pdb_folder,
        path=path,
        idr_regions=current_idrs,
        motifs=current_motifs,
        pae_threshold=1.0
    )
    export_matrices(ID, pae, dmaps_std)

In [ ]:
import os
import zipfile
from IPython.display import FileLink

# 1. Define the name of your output zip file
zip_name = "simulation_results.zip"

# 2. Create the zip file
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('/kaggle/working'):
        for file in files:
            # Avoid zipping the zip file itself if it's already there
            if file != zip_name:
                zipf.write(os.path.join(root, file), 
                           os.path.relpath(os.path.join(root, file), '/kaggle/working'))

# 3. Create a clickable download link
FileLink(zip_name)